In [5]:

# MTA – Aufschreibung bereinigen
#
# Bereinigung & Vereinheitlichung (Spalten, Datum/Zeit,
# Station/OP-Splitting, Freitext).
#
# Version angepasst für:
# - mehrere Excel-Dateien
# - automatisches Zusammenführen
# - Sheet "Aufschreibung"

import re
import unicodedata
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from pathlib import Path
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================================================
# DATEIEN
# =========================================================

SHEET_NAME = "Aufschreibung"

basispfad = Path(r"../data/raw/mta2024to2026")

dateien_liste = [
    basispfad / "STW-Mittelteilanlage 2024.xlsx",
    basispfad / "Störliste STW-Mittelteilanlage 2025.xlsx",
    basispfad / "Störliste STW-Mittelteilanlage 2026.xlsx"
]

# =========================================================
# 1) Spalten / Text Helpers
# =========================================================

def _normalize_colname(c: object) -> str:
    c = "" if c is None else str(c)
    c = unicodedata.normalize("NFKC", c)
    c = c.replace("\n", " ")
    c = re.sub(r"\s+", " ", c).strip()
    c = re.sub(r"[‐-‒–—―]", "-", c)
    c = re.sub(r"\s*/\s*", "/ ", c)
    c = re.sub(r"\s+", " ", c).strip()
    return c


_CANON_PATTERNS = [
    (r"^station\s*/\s*op$", "Station/ OP"),
    (r"^station/op$", "Station/ OP"),
    (r"^datum\s*neu$", "DatumNEU"),
    (r"^zeit\s*von$", "Zeit von"),
    (r"^zeit\s*bis$", "Zeit bis"),
    (r"^unterbrechungsursache$", "Unterbrechungsursache"),
    (r"^bemerkung$", "Bemerkung"),
    (r"^dauer\s*org-?\s*mangel$", "Dauer Org-Mangel"),
]


def canonicalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = [_normalize_colname(c) for c in df.columns]

    canon = []

    for c in cols:
        c2 = c

        for pat, repl in _CANON_PATTERNS:
            if re.match(pat, c2, flags=re.IGNORECASE):
                c2 = repl
                break

        canon.append(c2)

    seen = {}
    out = []

    for c in canon:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")

    df = df.copy()
    df.columns = out

    return df


def find_col(cols, patterns) -> str | None:

    for pat in patterns:
        for c in cols:
            if re.match(pat, c, flags=re.IGNORECASE):
                return c

    for c in cols:
        lc = c.lower()

        if "station" in lc and "op" in lc:
            return c

    return None


def normalize_free_text(s: pd.Series) -> pd.Series:

    s = s.astype("string")

    s = s.map(
        lambda x: unicodedata.normalize("NFKC", x)
        if pd.notna(x)
        else x
    )

    s = s.str.lower()

    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.replace(r"[‐-‒–—―]", "-", regex=True)
    s = s.str.replace(r"[•·●]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()

    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "none": pd.NA,
        "k.a.": pd.NA,
        "k. a.": pd.NA
    })

    return s


def fuzzy_standardize(
        norm_s: pd.Series,
        threshold: int = 97,
        min_count: int = 2
):

    counts = norm_s.dropna().value_counts()
    variants = counts.index.tolist()

    mapping = {}

    for v in variants:

        if v in mapping:
            continue

        mapping[v] = v

        matches = process.extract(
            v,
            variants,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold,
            limit=None
        )

        for m, score, _ in matches:
            if m not in mapping:
                mapping[m] = v

    std = norm_s.map(mapping).astype("string")
    std = std.where(norm_s.notna(), pd.NA)

    map_df = pd.DataFrame({
        "original": list(mapping.keys()),
        "standard": list(mapping.values()),
        "count": [counts.get(k, 0) for k in mapping.keys()]
    })

    map_df = map_df.sort_values(
        ["standard", "count"],
        ascending=[True, False]
    )

    if min_count > 1:
        map_df = map_df[map_df["count"] >= min_count].copy()

    return std, map_df


# =========================================================
# 2) Datum / Zeit Helpers
# =========================================================

def parse_excel_date_to_date(s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    return dt.dt.normalize()


def parse_excel_time_to_str(s: pd.Series) -> pd.Series:

    def conv(x):

        if pd.isna(x):
            return pd.NA

        if isinstance(x, datetime.time):
            return x.strftime("%H:%M:%S")

        if isinstance(x, (pd.Timestamp, datetime.datetime)):
            return x.time().strftime("%H:%M:%S")

        if isinstance(x, (float, int, np.floating, np.integer)):

            seconds = int(round(float(x) * 24 * 3600)) % (24 * 3600)

            h = seconds // 3600
            m = (seconds % 3600) // 60
            sec = seconds % 60

            return f"{h:02d}:{m:02d}:{sec:02d}"

        txt = str(x).strip()

        if not txt:
            return pd.NA

        t = pd.to_datetime(txt, errors="coerce")

        if pd.isna(t):
            return pd.NA

        return t.time().strftime("%H:%M:%S")

    return s.map(conv).astype("string")


def time_str_to_minutes(s: pd.Series) -> pd.Series:

    def conv(x):

        if pd.isna(x):
            return np.nan

        parts = str(x).split(":")

        if len(parts) < 2:
            return np.nan

        h = int(parts[0])
        m = int(parts[1])
        sec = int(parts[2]) if len(parts) > 2 else 0

        return h * 60 + m + sec / 60

    return s.map(conv).astype(float)


# =========================================================
# 3) Cleaning Funktion
# =========================================================

def bereinige_datensatz(df_raw: pd.DataFrame) -> pd.DataFrame:

    df = canonicalize_columns(df_raw)

    station_col = find_col(df.columns, [r"^Station/\s*OP$"])

    if station_col is None:
        raise ValueError(
            "Spalte 'Station/ OP' nicht gefunden."
        )

    date_col = find_col(df.columns, [r"^DatumNEU$"])
    t_from_col = find_col(df.columns, [r"^Zeit von$"])
    t_to_col = find_col(df.columns, [r"^Zeit bis$"])

    # Datum
    if date_col:
        df[date_col] = parse_excel_date_to_date(df[date_col])

    # Zeit von
    if t_from_col:
        df[t_from_col] = parse_excel_time_to_str(df[t_from_col])
        df["Zeit_von_min"] = time_str_to_minutes(df[t_from_col])

    # Zeit bis
    if t_to_col:
        df[t_to_col] = parse_excel_time_to_str(df[t_to_col])
        df["Zeit_bis_min"] = time_str_to_minutes(df[t_to_col])

    # Freitext vereinheitlichen
    for free_col in [
        "Bemerkung",
        "Unterbrechungsursache"
    ]:

        if free_col in df.columns:

            df[f"{free_col}_norm"] = normalize_free_text(
                df[free_col]
            )

            df[f"{free_col}_std"], map_df = fuzzy_standardize(
                df[f"{free_col}_norm"],
                threshold=97,
                min_count=2
            )

    return df


# =========================================================
# ALLE DATEIEN VERARBEITEN
# =========================================================

bereinigte_datenframes = []

for datei in dateien_liste:

    print(f"Verarbeite Datei: {datei}")

    df_raw = pd.read_excel(
        datei,
        sheet_name=SHEET_NAME
    )
   # print(datei.name, df_raw.shape)
    df_bereinigt = bereinige_datensatz(df_raw)

    bereinigte_datenframes.append(df_bereinigt)
   

# =========================================================
# ZUSAMMENFÜHREN
# =========================================================

df_gesamt = pd.concat(
    bereinigte_datenframes,
    axis=0,
    ignore_index=True
)

# =========================================================
# SORTIEREN
# =========================================================

if "DatumNEU" in df_gesamt.columns:

    df_gesamt = (
        df_gesamt
        .sort_values(by="DatumNEU")
        .reset_index(drop=True)
    )

print(df_gesamt.head())

print("\nGesamtgröße:", df_gesamt.shape)

# =========================================================
# EXPORT
# =========================================================

OUT_CSV = r"../data/raw/mta2024to2026/aufschreibung_mta_clean_2024to2026.csv"

OUT_XLSX = r"../data/raw/mta2024to2026/aufschreibung_mta_clean_2024to2026.xlsx"

df_gesamt.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8"
)

df_gesamt.to_excel(
    OUT_XLSX,
    index=False
)

print("\nGespeichert:")
print(OUT_CSV)
print(OUT_XLSX)


Verarbeite Datei: ..\data\raw\mta2024to2026\STW-Mittelteilanlage 2024.xlsx
Verarbeite Datei: ..\data\raw\mta2024to2026\Störliste STW-Mittelteilanlage 2025.xlsx
Verarbeite Datei: ..\data\raw\mta2024to2026\Störliste STW-Mittelteilanlage 2026.xlsx


C:\Users\golde\AppData\Local\Temp\ipykernel_12124\3166753531.py:342: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_gesamt = pd.concat(


       Datum Wochentag   DatumNEU                   KW       Std.        Log Schicht  Zeit von  Zeit bis  Dauer Arbeits-zeit  Anzahl MA  Menge N.i. O. Menge i. O. L4  Menge i. O. L5  \
0        NaT         5 2023-01-20  2023-01-21 00:00:00 2023-01-22 2023-01-23     NaN      <NA>      <NA>                 NaN        NaN            NaN            NaN             NaN   
1 2024-06-25         3 2023-03-29              2023/13        NaT        NaT     NaN      <NA>      <NA>                 NaN        NaN            NaN            NaN             NaN   
2 2024-06-25         3 2023-03-29              2023/13        NaT        NaT     NaN      <NA>      <NA>                 NaN        NaN            NaN            NaN             NaN   
3 2024-06-25         3 2023-03-29              2023/13        NaT        NaT       f  11:00:00  12:00:00                60.0        5.0            NaN            NaN             NaN   
4 2024-06-25         3 2023-03-29              2023/13        NaT        Na